# Multi-Agent Health & Nutrition Coaching System
**Architecture: LangGraph Orchestration & Local LLaMA 3 via Ollama**


## Overview & System Objectives
This project implements an autonomous **Multi-Agent System (MAS)** orchestrated with **LangGraph**, designed to generate coherent, personalized fitness and nutrition plans combining athletic conditioning, medical constraint handling, and dietary macronutrient optimization.

### Key Problem Formulation
Single monolithic LLMs frequently suffer from:
1. **Mathematical hallucinations** when computing caloric deficits, metabolic basal rates, and exact macro splits.
2. **Context degradation** when concurrently balancing complex physical contraindications (e.g., cardiovascular pathologies, asthma) and strict dietary restrictions (e.g., celiac disease, lactose intolerance).

### Multi-Agent Solution Architecture
Our architecture distributes domain responsibilities across specialized, cooperative agents:
- **Client Ingestion Agent**: Interactive profile intake and goal parsing.
- **Planner / Router Agent**: Intent decomposition and conditional workflow routing.
- **Specialized Athletic Coaches**: Dedicated domain agents for Running, Strength/Bodybuilding, Cycling, and Swimming.
- **Medical & Physical Constraint Guardian (`acont`)**: Parallel check against declared pathologies.
- **Nutrition & Dietary Specialist**: Caloric target synthesis leveraging verified empirical food datasets (`Calories.csv`, `cleaned_nutrition_dataset_per100g.csv`).
- **Safety Critic & Quality Guardrail**: Strict bidirectional verification loop capable of triggering revisions if caloric targets or safety guidelines are breached.


### Step 1: Environment Setup & Ollama Engine
Ensure [Ollama](https://ollama.com/) is installed locally and the LLaMA 3 model is pulled:
```bash
ollama run llama3
```


In [ ]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (544 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 121689 files and directories currently i

### Step 2: Background Ollama Server
Ensure the local Ollama daemon is active and listening on `http://localhost:11434`.


In [ ]:
import os
import subprocess
import time

# Lancer le serveur Ollama en arrière-plan
subprocess.Popen(["ollama", "serve"])

# Attendre que le serveur démarre
time.sleep(5)

# Télécharger le modèle (Llama 3 est performant pour le coaching)
!ollama pull llama3


### Step 3: Install Python Dependencies


In [ ]:
!pip install -U langgraph langchain_community langchain_ollama pandas rapidfuzz


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 135.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source o

### Step 4: Multi-Agent State Definition (`AgentState`)
Defines the shared state schema passing structured context across graph nodes.


In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3")

class AgentState(TypedDict):
    user_data: dict
    current_step: int
    is_complete: bool

    # --- Section Sport ---
    sport_plan: str
    e_cal: int
    # Nouveaux champs pour la validation Sport
    sport_feedback: Optional[str]
    sport_revision_count: int
    is_sport_valid: bool

    # --- Section Nutrition ---
    wants_nutrition: bool
    nutrition_plan: str
    has_constraints: bool
    constraints_report: str
    # Nouveaux champs pour la validation Nutrition
    nutrition_feedback: Optional[str]
    nutrition_revision_count: int
    is_nutrition_valid: bool

    router_decision: dict  # Contiendra : {"sport": "as_run", "check_nutrition": True, "check_health": True}


### Step 5: Empirical Dataset Ingestion (Food & Energy Expenditure)
Loads verified nutritional databases and MET activity tables for deterministic caloric calculations.


In [ ]:
from rapidfuzz import process
from rapidfuzz.process import extractOne

# Local dataset path resolution et n'as pas besoin d'être connecté à Google Drive
# Local dataset discovery
from pathlib import Path
base_data_dir = Path('../data') if Path('../data').exists() else Path('data')

import pandas as pd
import numpy as np
import os

# --- CHARGEMENT DES FICHIERS ---
def setup_data():
    # Load sport expenditure dataset
    df_s = pd.read_csv(str(base_data_dir / 'Calories.csv')) # A modifier avec votre lien vers le bon répertoire de fichier
    df_s.columns = [c.strip() for c in df_s.columns]

    # Weight column numeric conversions
    weight_cols = ['50 kg (110 lb)', '60 kg (132 lb)', '70 kg (154 lb)',
                   '80 kg (176 lb)', '90 kg (198 lb)', '100 kg (220 lb)', '120 kg (265 lb)']

    for col in weight_cols:
        if col in df_s.columns:
            # Replace decimal commas with dots and convert to float
            df_s[col] = pd.to_numeric(df_s[col].astype(str).str.replace(',', '.'), errors='coerce').fillna(0)

    # Load nutrition food dataset
    df_f = pd.read_csv(str(base_data_dir / 'cleaned_nutrition_dataset_per100g.csv'))  # A modifier avec votre lien vers le bon répertoire de fichier
    df_f.columns = [c.strip() for c in df_f.columns]

    return df_s, df_f

df_sport, df_food = setup_data()

# Clean spurious outliers in nutrition dataset (pour ne pas avoir de valeurs aberrantes)
def clean_nutrition_outliers(df):
    # Un aliment ne peut pas avoir plus de 100g de macros pour 100g de produit
    cols_to_fix = ['Protein (g per 100g)', 'Fat (g per 100g)', 'Carbohydrates (g per 100g)']
    for col in cols_to_fix:
        # On supprime les lignes aberrantes ou on les plafonne à 30g (valeur moyenne haute)
        df = df[df[col] <= 100]
    return df

df_food = clean_nutrition_outliers(df_food)

# --- CALCULATEUR DE CALORIES ---
def get_real_calories(sport_query, weight_kg, level, duration=60):
    # 1. Trouver le sport le plus proche
    activities = (df_sport['Activity'].fillna('') + " " + df_sport['Subtype'].fillna('')).tolist()
    match, score, idx = extractOne(sport_query, activities)

    # 2. Trouver la colonne de poids la plus proche
    user_weight = float(weight_kg)
    available_weights = [50, 60, 70, 80, 90, 100, 120]
    closest_kg = min(available_weights, key=lambda x: abs(x - user_weight))

    # Construction du nom de la colonne (ex: "70 kg (154 lb)")
    lb_val = int(closest_kg * 2.2)
    column_name = f"{closest_kg} kg ({lb_val} lb)"

    # 3. Récupération de la valeur numérique (déjà convertie en float lors du setup)
    base_calories = float(df_sport.iloc[idx][column_name])

    # 4. Multiplicateur de NIVEAU (en fonction du nivau renseigné par l'utilisateur)
    # Avancé = plus de puissance/intensité (1.3) | Débutant = intensité modérée (0.8)
    level_map = {"débutant": 0.8, "intermédiaire": 1.0, "avancé": 1.3}
    mult = level_map.get(level.lower(), 1.0)

    total = (base_calories * (duration / 60)) * mult
    return int(total), match

# --- OUTIL : RECHERCHE NUTRITIONNELLE ---
def get_food_context(objective, limit=8):
    # Tri intelligent selon l'objectif (prise de masse : tri en fonction des proteines | autres objectifs : tri en fonction du nombre de calories)
    if "masse" in objective.lower():
        data = df_food.sort_values(by='Protein (g per 100g)', ascending=False)
    else:
        data = df_food.sort_values(by='Calories (kcal per 100g)', ascending=True)
    # On recupere seulement les colonnes utiles pour notre projet pour éviter de surcharger avec des données inutiles pour nos besoins
    return data.head(limit)[['food', 'Calories (kcal per 100g)', 'Protein (g per 100g)', 'Fat (g per 100g)', 'Carbohydrates (g per 100g)']].to_markdown()


Mounted at /content/drive


### Step 6: Specialized Agent Implementation
Implementation of domain-specific agents, prompts, and tool integrations.


In [ ]:
import time
import json

# --- CLIENT AGENT (Interactive Profile Collection) ---
# --- CLIENT AGENT ---
def agent_client_interactive(state: AgentState):
    # --- Automated test guardrail ---
    # Bypass interactive questionnaire when executing automated tests
    if state.get("is_complete"):
        return {}
    # -----------------------------------

    questions = [
        ("nom", "Comment t'appelles-tu ?"),
        ("genre", "Quel est ton genre (Homme/Femme) ?"),
        ("poids", "Quel est ton poids actuel (en kg) ?"),
        ("objectif", "Quel est ton objectif (Sèche, Prise de masse, etc.) ?"),
        ("sport", "Sport préféré ? (Laisse vide pour que je choisisse pour toi)"),
        ("niveau", "Quel est ton niveau (Débutant, Intermédiaire, Avancé) ?"),
        ("frequence", "Combien de séances par semaine ?"),
        ("conditions_physiques", "As-tu des maladies ou conditions physiques (ex: problèmes cardiaques, blessures) ? (Sinon tape Non)"),
        ("wants_nutrition", "Veux-tu un programme nutritionnel associé ? (Oui/Non)"),
    ]

    step = state.get("current_step", 0)
    user_data = state.get("user_data", {})

    # Logique dynamique
    if "wants_nutrition" in user_data:
        wants_nutri = user_data["wants_nutrition"].lower() in ["oui", "o", "yes"]
        if wants_nutri:
            questions.append(("contraintes", "As-tu des allergies/maladies ? (Sinon tape 'Non')"))
        else:
            user_data["contraintes"] = "Non"

    if step < len(questions):
        if "contraintes" in user_data and user_data["contraintes"] == "Non":
             pass
        else:
            key, question = questions[step]
            user_response = input(f"🤖 [Agent Client]: {question}\n👤: ")
            user_data[key] = user_response
            return {"user_data": user_data, "current_step": step + 1, "is_complete": False}

    print("\n--- ✅ DONNÉES COLLECTÉES ---")

    # ... (Affichage résumé) ...
    for k, v in user_data.items():
        print(f"- {k.replace('_', ' ').capitalize()}: {v}")

    # C'est ICI que ça plantait :
    confirm = input("\n🤖 Est-ce correct pour lancer la génération du sport ? (O/N) : ")

    if confirm.lower() == 'o':
        wants_nutri = user_data["wants_nutrition"].lower() in ["oui", "o", "yes"]
        has_constraints = user_data.get("contraintes", "Non").lower() not in ["non", "aucun", "aucune", "ras"]

        return {"is_complete": True,
                "wants_nutrition": wants_nutri,
                "has_constraints": has_constraints,
                "user_data": user_data
                }
    else:
        return {"user_data": {}, "current_step": 0, "is_complete": False}
# --- AGENT DE RETRY ( si le sport renseigné par l'utilisateur n'est pas supporté dans la liste des sports du projet) ---
def agent_ask_retry_node(state: AgentState):
    user = state["user_data"]
    bad_sport = user.get("sport", "Inconnu")

    print(f"\n❌ [Système]: Le choix '{bad_sport}' n'est pas supporté ou incohérent.")
    print("   ⚠️ Pour garantir un bon programme, nous allons redéfinir tes paramètres sportifs.")
    print("   (Les experts dispos sont : Musculation, Running, Vélo, Natation)")

    # 1. On redemande l'OBJECTIF (car il peut changer avec le sport)
    user["objectif"] = input("\n🎯 Quel est ton objectif principal ? : ")

    # 2. On redemande le SPORT (ou laisser vide pour déduction)
    print("🏃 Quel sport choisis-tu ? (Laisse vide pour que je choisisse selon l'objectif)")
    user["sport"] = input("👤: ")

    # 3. On redemande le NIVEAU (car on peut être expert en Tennis mais débutant en Muscu)
    user["niveau"] = input("📊 Quel est ton niveau (Débutant/Intermédiaire/Avancé) ? : ")

    # 4. On redemande la FRÉQUENCE
    user["frequence"] = input("📅 Combien de séances par semaine ? : ")

    print("\n🔄 [Système]: Données mises à jour. Retour à l'analyse...")

    # On renvoie l'état complet mis à jour
    return {"user_data": user}

# --- AGENT PLANIFICATEUR (va permettre d'assigner dynamiquement la requête de l'utilisateur à l'agent correspondant au besoin et au sport)---
def agent_planner_node(state: AgentState):
    print("\n🧠 [Planificateur]: Analyse de la demande pour affectation de l'expert...")
    user = state["user_data"]
    # On nettoie l'entrée
    user_sport = user.get("sport", "").lower().strip()
    user_objectif = user.get("objectif", "Être en forme")

    # --- PHASE 1 : GARDE-FOU PYTHON (Validation Stricte) ---
    # On vérifie si l'entrée utilisateur correspond aux compétences

    valid_keywords = {
        "as_muscu": ["muscu", "poids", "force", "gym", "body", "crossfit", "fonte", "haltère", "masse"],
        "as_run": ["courir", "run", "marathon", "sprint", "cardio", "jogging", "course", "trail"],
        "as_cycle": ["vélo", "velo", "vtt", "cyclis", "biking", "triathlon"],
        "as_swim": ["nage", "piscine", "natation", "eau", "brasse", "crawl"]
    }

    # Si l'utilisateur a écrit quelque chose (n'est pas vide)
    if user_sport:
        is_known_sport = False
        # On cherche si un mot-clé est présent dans ce qu'il a écrit
        for agent_key, keywords in valid_keywords.items():
            if any(k in user_sport for k in keywords):
                is_known_sport = True
                break # C'est bon, on connait ce sport

        # SI C'EST INCONNU (ex: "Tennis", "Parachute") -> ON RENVOIE VERS LA CORRECTION
        if not is_known_sport:
            print(f"   ⚠️ Sport non reconnu : '{user_sport}'. Demande de correction.")
            return {
                "router_decision": {
                    "selected_sport_agent": "retry_input", # <--- Le signal pour le routeur
                    "activate_health_agent": False,
                    "activate_nutrition_loop": False
                }
            }

    # --- PHASE 2 : DÉDUCTION PAR L'IA (Si vide ou valide) ---
    # Si on arrive ici, c'est que soit le champ est vide (à déduire),
    # soit il est valide (à confirmer + options santé).

    if not user_sport:
        user_sport = "NON SPÉCIFIÉ (À DÉDUIRE DE L'OBJECTIF)"

    # On récupére le contexte pour envoyer les données à l'agent de sports / nutrition / contraintes par la suite
    context = f"""
    Profil: {user.get('nom')}, {user.get('genre')}
    Objectif: {user.get('objectif')}
    Sport souhaité par l'utilisateur: {user_sport}
    Contraintes médicales: {user.get('contraintes')}
    Demande Nutrition: {user.get('wants_nutrition')}
    """

    # Prompt de l'agent planificateur pour qu'il choisisse le bon sport et s'il n'y a pas de sport renseigné, il le déduit parmi la liste

    prompt = f"""
    Tu es le DISPATCHER du système de coaching.
    Ta mission est d'associer la demande utilisateur à l'un des 4 experts disponibles, OU de rejeter la demande.
    [CONTEXTE UTILISATEUR]
    {context}

    [LES 4 EXPERTS AUTORISÉS]
    1. "as_muscu" : UNIQUEMENT pour : Musculation, Poids, Haltères, Prise de masse, Force, Crossfit, Gym.
    2. "as_run"   : UNIQUEMENT pour : Course à pied, Marathon, Jogging, Sprint, Cardio, Endurance.
    3. "as_cycle" : UNIQUEMENT pour : Vélo, VTT, Cyclisme, Triathlon.
    4. "as_swim"  : UNIQUEMENT pour : Natation, Piscine, Nage.

    [RÈGLE DE REJET (TRES IMPORTANT)]
    Si la demande concerne un autre sport (Exemples : Parachute, Tennis, Foot, Judo, Danse, Ski, Golf, Echecs...) -> TU DOIS RÉPONDRE "error_sport".
    Ne cherche pas de lien indirect. Si ce n'est pas explicitement de la muscu, du running, du vélo ou de la nage : C'EST UNE ERREUR.

    [EXEMPLES DE DÉCISION (Follow these patterns)]
    - User: "Je veux courir vite" -> "as_run"
    - User: "Je veux faire du parachutisme" -> "error_sport"
    - User: "Apprendre à sauter en parachute" -> "error_sport"
    - User: "Jouer au tennis" -> "error_sport"
    - User: "Prendre du muscle" -> "as_muscu"

    [RÈGLES DE DÉCISION STRICTES]
    1. Si l'utilisateur demande un sport couvert (ex: "Je veux courir"), choisis l'expert correspondant.
    2. Si le sport est vide, DÉDUIS-LE de l'objectif (ex: "Perdre du poids" -> as_run ou as_muscu).

    [TA MISSION]
    1. Choisis le MEILLEUR expert sport (un seul).
    2. Active l'expert santé (activate_health) SI des contraintes/allergies sont détectées (sauf si "Non" ou "Rien").
    3. Active la nutrition (activate_nutrition) SI l'utilisateur a répondu Oui.


    [FORMAT DE RÉPONSE JSON OBLIGATOIRE]
    {{
        "selected_sport_agent": "nom_de_l_agent",
        "activate_health_agent": true/false,
        "activate_nutrition_loop": true/false
    }}
    Réponds UNIQUEMENT le JSON.
    """

    response = llm.invoke(prompt)

    # Parsing sécurisé (Llama 3 est parfois bavard)
    try:

        # Extraction JSON robuste
        start = response.find('{')
        end = response.rfind('}') + 1
        if start == -1: raise ValueError("No JSON")

        decision = json.loads(response[start:end])
        print(f"   📋 Ordre de mission en détail : {decision}")
    except:
        print("   ⚠️ Fallback sécurité (Erreur JSON).")

        # Si ça plante, on assume que c'est une erreur de demande
        decision = {"selected_sport_agent": "error_sport", "activate_health_agent": False}

    return {"router_decision": decision}

# --- RÉSEAUX D'AGENTS DE SPORT (RAS) ---

# --- AGENT DE MUSCULATION ---
def as_musculation_node(state: AgentState):
    start = time.time()
    print(f"\n⏱️ [START] Agent Sport (à {time.strftime('%H:%M:%S', time.localtime(start))}")

    revision = state.get("sport_revision_count", 0)
    feedback = state.get("sport_feedback", "")
    user = state["user_data"]

    # Calcul dynamique des calories
    e_cal_real, match = get_real_calories("Weight lifting", user['poids'], user['niveau'])

    # Gestion de la correction
    correction_instruction = ""
    if feedback and revision > 0:
        print(f"\n🏋️ [Expert Musculation]: ⚠️ Correction du programme (Tentative {revision+1})...")
        correction_instruction = f"""
        [MODE CORRECTION ACTIVÉ]
        Ton précédent programme a été REJETÉ pour cette raison : "{feedback}".
        Tu dois impérativement corriger ce point tout en respectant le format structurel ci-dessous.
        """
    else:
        print("\n🏋️ [Expert Musculation]: Génération du programme...")

    # Prompt avec structure de sortie imposée
    prompt = f"""
    [PERSONNALITÉ]
    Tu es un Coach de Musculation de haut niveau. Ton approche est scientifique et rigoureuse (hypertrophie, RPE, surcharge progressive).

    [CONTEXTE]
    Utilisateur : {user['genre']} | Niveau : {user['niveau']} | Objectif : {user['objectif']} | Fréquence : {user['frequence']} séances/semaine.
    Dépense calculée : {e_cal_real} kcal/h.

    {correction_instruction}

    [CONSIGNE DE FORMATAGE STRICTE]
    Tu dois générer le programme en respectant scrupuleusement le plan suivant.
    Utilise du Markdown (tableaux et listes) pour la lisibilité.
    NE FAIS PAS de longs paragraphes de texte.

    [CONSIGNE DE LANGUE OBLIGATOIRE] :
    - Tu dois répondre EXCLUSIVEMENT en français.
    - Il est strictement INTERDIT d'utiliser l'anglais, même pour les noms d'exercices ou les termes techniques.
    - Traduis tout : "Bench Press" devient "Développé couché", "Squat" reste "Squat" (car utilisé en français) mais les conseils doivent être en français.

    ### 1. RÉSUMÉ DU CYCLE
    * **Focus principal** : [Expliquer en 1 phrase]
    * **Intensité globale** : [RPE recommandé]

    ### 2. PLANNING HEBDOMADAIRE (Tableau)
    | Jour | Séance | Focus | Durée |
    | :--- | :--- | :--- | :--- |
    | Jour 1 | [Nom] | [Détail] | [XX min] |
    (Compléter pour {user['frequence']} jours)

    ### 3. DÉTAIL DES SÉANCES
    Pour chaque séance, utilise ce format :
    **[NOM DE LA SÉANCE]**
    * **Exercice 1** : [Nom] | [Séries] x [Répétitions] | [Repos]
        - *Technique* : [Note courte]
        - *Option Sans Machine* : [Alternative poids libre/corps]
    * **Exercice 2** : ...

    ### 4. STRATÉGIE D'ÉVOLUTION (SURCHARGE PROGRESSIVE)
    * **Semaine 1-2** : [Consigne]
    * **Semaine 3-4** : [Comment augmenter la difficulté]

    [CONTRAINTE TECHNIQUE ABSOLUE]
    Ton message doit se terminer par la mention exacte suivante :
    CALORIES: {e_cal_real}
    """

    response = llm.invoke(prompt)
    end = time.time()
    print(f"⏱️ [END] Agent Sport fini à {time.strftime('%H:%M:%S', time.localtime(end))} (Durée: {round(end-start, 2)}s)")

    return {"sport_plan": response, "e_cal": e_cal_real, "sport_revision_count": revision + 1}

def as_running_node(state: AgentState):

    # Mesure du temps de début de l'agent
    start = time.time()
    print(f"\n⏱️ [START] Agent Running (à {time.strftime('%H:%M:%S', time.localtime(start))}")

    # Récupération des compteurs et feedback
    revision = state.get("sport_revision_count", 0)
    feedback = state.get("sport_feedback", "")
    user = state["user_data"]

    # Calcul dynamique des calories
    e_cal_real, match = get_real_calories("Running", user['poids'], user['niveau'])

    # Logique de correction
    correction_instruction = ""
    if feedback and revision > 0:
        print(f"\n🏃 [Expert Running]: ⚠️ Correction du programme (Tentative {revision+1})...")
        correction_instruction = f"""
        [MODE CORRECTION ACTIVÉ]
        Ton précédent programme a été REJETÉ pour la raison suivante : "{feedback}".
        Corrige impérativement ce point tout en respectant scrupuleusement le format ci-dessous.
        """
    else:
        print("\n🏃 [Expert Running]: Génération du programme...")

    # Prompt avec structure de sortie imposée pour le Running
    prompt = f"""
    [PERSONNALITÉ]
    Tu es un Coach de Running de niveau international, expert en physiologie de l'effort et en planification (périodisation).

    [CONTEXTE]
    Utilisateur : {user['genre']} | Niveau : {user['niveau']} | Objectif : {user['objectif']} | Fréquence : {user['frequence']} séances/semaine.
    Dépense calculée : {e_cal_real} kcal/h.

    {correction_instruction}

    [CONSIGNE DE FORMATAGE STRICTE]
    Génère un plan d'entraînement évolutif. Ne fais pas de longs blocs de texte. Utilise des tableaux et des listes.


    [CONSIGNE DE LANGUE ET TERMINOLOGIE RUNNING OBLIGATOIRE] :
    - Tu dois répondre EXCLUSIVEMENT en français.
    - TRADUCTION DES MÉTRIQUES : N'utilise pas "Pace", utilise "Allure" (en min/km). N'utilise pas "Heart Rate", utilise "Fréquence Cardiaque".
    - VOCABULAIRE TECHNIQUE : Utilise les termes officiels du running français :
        * "Endurance Fondamentale" (au lieu de Slow Run)
        * "Fractionné" ou "Séance de VMA" (au lieu de Interval training)
        * "Seuil" (au lieu de Threshold)
        * "Sortie longue" (au lieu de Long run)
        * "Échauffement" et "Retour au calme" (au lieu de Warm-up / Cool-down).
    - Les conseils de posture et de respiration doivent être rédigés dans un français technique et motivant.

    ### 1. STRATÉGIE DE PROGRESSION (Périodisation)
    * **Phase actuelle** : [Ex: Base, Puissance ou Affûtage]
    * **Objectif de progression** : [Comment le volume (km) et l'intensité évoluent vers {user['objectif']}]

    ### 2. PLANNING HEBDOMADAIRE (Tableau)
    | Jour | Type de Séance | Objectif | Durée/Distance |
    | :--- | :--- | :--- | :--- |
    | Jour 1 | [Ex: Endurance F.] | [Détail technique] | [XX min / XX km] |
    (Répète pour les {user['frequence']} jours prévus)

    ### 3. DÉTAIL TECHNIQUE DES SÉANCES
    Pour chaque séance, précise :
    **[NOM DE LA SÉANCE]**
    * **Description** : [Explication courte du but physiologique]
    * **Allure/Rythme** : [min/km ou zone de fréquence cardiaque adaptée au niveau {user['niveau']}]
    * **Option Tapis/Intérieur** : [Variante si météo défavorable]

    ### 4. RENFORCEMENT (PPG)
    * **Exercices ciblés** : [Gainage, proprioception, fessiers]
    * **Fréquence intégrée** : [Quand le réaliser dans la semaine]

    [CONTRAINTE TECHNIQUE ABSOLUE]
    Ton message doit se terminer par la mention exacte suivante, sans aucun texte après :
    CALORIES: {e_cal_real}
    """

    # Appel du LLM
    response = llm.invoke(prompt)

    # Fin du compteur
    end = time.time()
    print(f"⏱️ [END] Agent Running fini à {time.strftime('%H:%M:%S', time.localtime(end))} (Durée: {round(end-start, 2)}s)")

    return {"sport_plan": response, "e_cal": e_cal_real, "sport_revision_count": revision + 1}

def as_cyclisme_node(state: AgentState):

    # Mesure du temps de début de l'agent
    start = time.time()
    print(f"\n⏱️ [START] Agent Cyclisme (à {time.strftime('%H:%M:%S', time.localtime(start))}")

    # Récupération des compteurs et feedback
    revision = state.get("sport_revision_count", 0)
    feedback = state.get("sport_feedback", "")
    user = state["user_data"]

    # Calcul dynamique des calories
    e_cal_real, match = get_real_calories("Cycling", user['poids'], user['niveau'])

    # Logique de correction
    correction_instruction = ""
    if feedback and revision > 0:
        print(f"\n🚴 [Expert Cyclisme]: ⚠️ Correction du programme (Tentative {revision+1})...")
        correction_instruction = f"""
        [MODE CORRECTION ACTIVÉ]
        Ton précédent programme a été REJETÉ pour la raison suivante : "{feedback}".
        Tu dois impérativement corriger ce point tout en respectant scrupuleusement le format ci-dessous.
        """
    else:
        print("\n🚴 [Expert Cyclisme]: Génération du programme...")

    # Prompt avec structure de sortie imposée pour le Cyclisme
    prompt = f"""
    [PERSONNALITÉ]
    Tu es un Directeur Sportif et Coach de Cyclisme de haut niveau, expert en sciences de l'endurance et en optimisation de la puissance (FTP). Ton ton est celui d'un mentor technique et motivant.

    [CONTEXTE]
    Utilisateur : {user['genre']} | Niveau : {user['niveau']} | Objectif : {user['objectif']} | Fréquence : {user['frequence']} séances/semaine.
    Dépense calculée : {e_cal_real} kcal/h.

    {correction_instruction}

    [CONSIGNE DE FORMATAGE STRICTE]
    Tu dois générer le programme en respectant scrupuleusement le plan suivant.
    Utilise du Markdown (tableaux et listes) pour la lisibilité.
    NE FAIS PAS de longs paragraphes de texte.


    [CONSIGNE DE LANGUE ET TERMINOLOGIE CYCLISME OBLIGATOIRE] :
    - Tu dois répondre EXCLUSIVEMENT en français.
    - UNITÉS ET MÉTRIQUES : Utilise "Allure" ou "Vitesse" (km/h), "Cadence" (en RPM - tours par minute), et "Puissance" (en Watts).
    - TRADUCTION TECHNIQUE : Utilise les termes officiels du cyclisme francophone :
        * "Foncier" ou "Endurance de base" (au lieu de Base ride).
        * "Fractionné" ou "Intervalles" (au lieu de Intervals).
        * "Vélocité" (travail de la cadence élevée).
        * "Force sous-max" (travail à basse cadence).
        * "Danseuse" (au lieu de Standing position).
        * "Bec de selle" (position vers l'avant).
    - HOME-TRAINER : Pour les alternatives intérieures, parle de "Séance sur Home-trainer" et non de "Indoor cycling".
    - SÉCURITÉ : Les conseils sur le réglage de la selle ou la position doivent être précis et en français.


    ### 1. STRATÉGIE DE PERFORMANCE (Périodisation)
    * **Phase actuelle** : [Ex: Foncier, Seuil ou Puissance]
    * **Objectif technique** : [Expliquer le travail sur la FTP ou l'endurance spécifique pour {user['objectif']}]

    ### 2. PLANNING HEBDOMADAIRE (Tableau)
    | Jour | Type de Séance | Intensité Cible | Durée |
    | :--- | :--- | :--- | :--- |
    | Jour 1 | [Ex: Intervalles] | [Zone de puissance/RPE] | [XX min] |
    (Répète pour les {user['frequence']} jours prévus)

    ### 3. DÉTAIL TECHNIQUE DES SÉANCES
    Pour chaque séance, précise :
    **[NOM DE LA SÉANCE]**
    * **Structure** : [Détail : Échauffement / Corps de séance / Récupération]
    * **Cadence Cible** : [Ex: 85-95 RPM]
    * **Alternative Home-Trainer** : [Adaptation spécifique pour l'intérieur]

    ### 4. TECHNIQUE & POSTURE
    * **Drill de la semaine** : [Ex: Pédalage unijambiste ou travail de vélocité]
    * **Note de sécurité** : [Conseil court sur la position ou le réglage du vélo]

    [CONTRAINTE TECHNIQUE ABSOLUE]
    Ton message doit se terminer par la mention exacte suivante, sans aucun texte après :
    CALORIES: {e_cal_real}
    """

    # Appel du LLM
    response = llm.invoke(prompt)

    # Fin du compteur
    end = time.time()
    print(f"⏱️ [END] Agent Cyclisme fini à {time.strftime('%H:%M:%S', time.localtime(end))} (Durée: {round(end-start, 2)}s)")

    return {"sport_plan": response, "e_cal": e_cal_real, "sport_revision_count": revision + 1}

def as_natation_node(state: AgentState):

    # Mesure du temps de début
    start = time.time()
    print(f"\n⏱️ [START] Agent Natation (à {time.strftime('%H:%M:%S', time.localtime(start))}")

    # Récupération des compteurs et feedback
    revision = state.get("sport_revision_count", 0)
    feedback = state.get("sport_feedback", "")
    user = state["user_data"]

    # Calcul dynamique des calories
    e_cal_real, match = get_real_calories("Swimming", user['poids'], user['niveau'])

    # Logique de correction
    correction_instruction = ""
    if feedback and revision > 0:
        print(f"\n🏊 [Expert Natation]: ⚠️ Correction du programme (Tentative {revision+1})...")
        correction_instruction = f"""
        [MODE CORRECTION ACTIVÉ]
        Ton précédent programme a été REJETÉ pour la raison suivante : "{feedback}".
        Tu dois impérativement corriger ce point tout en respectant scrupuleusement le format ci-dessous.
        """
    else:
        print("\n🏊 [Expert Natation]: Génération du programme...")

    # Prompt avec structure de sortie imposée pour la Natation
    prompt = f"""
    [PERSONNALITÉ]
    Tu es un Entraîneur de Natation de haut niveau, expert en hydrodynamique et technique de nage. Ton ton est discipliné, précis et pédagogique.

    [CONTEXTE]
    Utilisateur : {user['genre']} | Niveau : {user['niveau']} | Objectif : {user['objectif']} | Fréquence : {user['frequence']} séances/semaine.
    Dépense calculée : {e_cal_real} kcal/h.

    {correction_instruction}

    [CONSIGNE DE FORMATAGE STRICTE]
    Tu dois générer le programme en respectant scrupuleusement le plan suivant.
    Utilise du Markdown (tableaux et listes) pour la lisibilité.
    NE FAIS PAS de longs paragraphes de texte.


    [CONSIGNE DE LANGUE ET TERMINOLOGIE NATATION OBLIGATOIRE] :
    - Tu dois répondre EXCLUSIVEMENT en français.
    - STRUCTURE DE SÉANCE : Utilise les termes officiels des bassins français :
        * "Échauffement" (au lieu de Warm-up).
        * "Éducatifs" (au lieu de Drills).
        * "Corps de séance" ou "Série principale" (au lieu de Main set).
        * "Récupération" ou "Retour au calme" (au lieu de Cool-down).
    - TECHNIQUES ET MATÉRIEL :
        * Utilise "Crawl", "Dos crawlé", "Brasse" et "Papillon".
        * Utilise "4 nages" (au lieu de IM ou Individual Medley).
        * Pour le matériel, dis : "Planche", "Pull-buoy", "Plaquettes", "Palmes".
        * "Amplitude" (au lieu de Stroke length).
        * "Culbute" (au lieu de Flip turn).
    - MÉTRIQUES : Toutes les distances doivent être en mètres (m). Les repos sont en secondes ou minutes.


    ### 1. FOCUS TECHNIQUE DE LA SEMAINE
    * **Objectif principal** : [Ex: Amplitude, Respiration ou Vitesse]
    * **Volume total estimé** : [Distance totale en mètres sur la semaine]

    ### 2. PLANNING HEBDOMADAIRE (Tableau)
    | Jour | Type de Séance | Focus | Distance Totale |
    | :--- | :--- | :--- | :--- |
    | Jour 1 | [Ex: Endurance] | [Détail technique] | [XXXX m] |
    (Répète pour les {user['frequence']} jours prévus)

    ### 3. DÉTAIL DES SÉANCES (Format Club)
    Pour chaque séance, utilise cette structure :
    **[NOM DE LA SÉANCE]**
    * **Échauffement** : [Distance/Exercice]
    * **Éducatifs (Drills)** : [Ex: Rattrapé, poings fermés - Expliquer l'utilité]
    * **Corps de séance** : [Séries x Distance | Repos | Intensité]
    * **Récupération** : [Nage souple]

    ### 4. ALTERNATIVE HORS DE L'EAU (DRY-LAND)
    * **Renforcement** : [Exercices avec élastiques ou gainage si accès piscine impossible]
    * **Conseil technique** : [Note sur le placement du regard ou du coude]

    [CONTRAINTE TECHNIQUE ABSOLUE]
    Ton message doit se terminer par la mention exacte suivante, sans aucun texte après :
    CALORIES: {e_cal_real}
    """

    # Appel du LLM
    response = llm.invoke(prompt)

    # Fin du compteur
    end = time.time()
    print(f"⏱️ [END] Agent Natation fini à {time.strftime('%H:%M:%S', time.localtime(end))} (Durée: {round(end-start, 2)}s)")

    return {"sport_plan": response, "e_cal": e_cal_real, "sport_revision_count": revision + 1}

# --- AGENT DE CONTRAINTES ---
def acont_node(state: AgentState):

    # Collecte des détails

    #On fait une mesure du temps de début de l'agent (pour vérifier le parallélisme par la suite et calculer le temps d'exécution de l'agent)
    start = time.time()
    print(f"\n⏱️ [START] Agent Contraintes à {time.strftime('%H:%M:%S', time.localtime(start))}")

    user_data = state["user_data"]
    # Nettoyage de la donnée pour la vérification
    details_raw = user_data.get("contraintes", "Aucune")
    details_clean = details_raw.strip().lower()

    # Si l'utilisateur n'a pas de contraintes, on ne sollicite PAS l'IA.
    if details_clean in ["non", "aucun", "aucune", "ras", "rien", "n/a"]:
        print(f"⚠️ [Agent Contraintes]: Aucune restriction détectée. Passage en mode sain.")

        # On renvoie un rapport fixe sans appeler llm.invoke
        return {
            "constraints_report": "AUCUNE RESTRICTION : L'utilisateur n'a déclaré aucune allergie ni maladie."
        }

    print(f"\n⚠️ [Agent Contraintes]: Analyse en parallèle des restrictions : '{details_raw}'...")

    # Configuration du prompt en utilisant les informations saisies par l'utilisateur
    prisme_prompt = f"""
    [PERSONNALITÉ] : Tu es un Expert en Nutrition Clinique et Diététicien spécialisé en allergologie.

    [RESTE / CONTEXTE] : Un client souhaite adapter son alimentation. Voici ses contraintes spécifiques : "{details_raw}".

    [INSTRUCTION] : Analyse précisément ces contraintes pour identifier les risques alimentaires et proposer des solutions sûres.

    [SPÉCIFICATIONS] : Produis un rapport structuré avec :
    1. Une liste des ingrédients ou familles d'aliments strictement INTERDITS.
    2. Une liste de SUBSTITUTIONS intelligentes et nutritives pour compenser les évictions.
    3. Un conseil de sécurité court.
    Utilise un ton professionnel et rassurant.

    [MISES EN GARDE] : Ne propose aucune substance pouvant interagir négativement avec les pathologies citées. Si une contrainte est floue, demande de consulter un médecin. Évite les conseils génériques.

    [EXEMPLE / ÉVALUATION] : Le résultat doit être directement utilisable par un chef cuisinier pour préparer un repas sans risque pour le client.
    """

    # Appel du LLM
    response = llm.invoke(prisme_prompt)

    # Ici on arrête le compteur pour avoir le temps total d'utilisation de l'agent
    end = time.time()
    print(f"⏱️ [END] Agent Contraintes fini à {time.strftime('%H:%M:%S', time.localtime(end))} (Durée: {round(end-start, 2)}s)")

    return {"constraints_report": response}

# --- AGENT NUTRITION ---
def an_nutrition_node(state: AgentState):

    # S'il n'y a pas eu de demande de programme de nutrition, on arrête l'agent
    if not state.get("wants_nutrition"):
        print("\n🏁 [Système]: Pas de nutrition demandée.")
        return {"nutrition_plan": "Non demandé", "is_nutrition_valid": True}

    # --- SYNCHRONISATION ---
    sport_is_done = state.get("is_sport_valid") or state.get("sport_revision_count", 0) >= 3
    if not sport_is_done:
        return None

    if state.get("has_constraints") and not state.get("constraints_report"):
        return None

    # --- PRÉPARATION DES DONNÉES ---
    constraints = state.get("constraints_report", "Aucune restriction spécifique.") if state.get("has_constraints") else "Aucune"
    revision = state.get("nutrition_revision_count", 0)
    feedback = state.get("nutrition_feedback", "")
    cal_sport = state["e_cal"]
    user = state["user_data"]
    food_data = get_food_context(user['objectif'])

    # --- LOGIQUE MÉTIER : CALCUL DE LA CIBLE CALORIQUE ---
    # Calcul d'une maintenance de base (estimée) + dépense sport + ajustement objectif
    base_cal = 2400 if user['genre'].lower() == "homme" else 1900
    # Prise de masse = +500 kcal, Sèche = -500 kcal
    goal_adj = 500 if "prise" in user['objectif'].lower() else (-500 if "sèche" in user['objectif'].lower() else 0)
    total_target = base_cal + cal_sport + goal_adj

    # Gestion de la correction
    correction_instruction = ""
    if feedback and revision > 0:
        print(f"\n🥗 [Agent Nutrition]: ⚠️ Correction du menu (Tentative {revision+1})...")
        correction_instruction = f"""
        [MODE CORRECTION ACTIVÉ]
        Le plan précédent a été rejeté car : "{feedback}".
        Génère un nouveau plan corrigeant cela, tout en respectant le format strict ci-dessous.
        """
    else:
        print("\n🥗 [Agent Nutrition]: Génération du plan...")

    # 4. Le Prompt Optimisé
    prisme_prompt = f"""
    [PERSONNALITÉ]
    Tu es un Nutritionniste du Sport et Chef de cuisine expert. Ton approche allie précision scientifique et plaisir gustatif.

    [CONTEXTE & SÉCURITÉ]
    - Utilisateur : {user['genre']} | Poids : {user['poids']} kg | Objectif : {user['objectif']}
    - CIBLE ÉNERGÉTIQUE : {total_target} kcal/jour (calculée : base + sport + objectif).
    - CONTRAINTES STRICTES : {constraints}
    - DATASET RÉEL : {food_data}

    {correction_instruction}

    [CONSIGNE DE SÉLECTION DES ALIMENTS]
    1. Priorise UNIQUEMENT des aliments simples, habituels et faciles à trouver en supermarché.
    Sauf demande explicite d'exotisme, utilise en priorité :
    - Protéines : Poulet, Dinde, Bœuf haché 5%, Œufs, Thon, Pavé de Saumon, Fromage blanc, Jambon.
    - Féculents : Riz, Pâtes, Pommes de terre, Patate douce, Flocons d'avoine, Pain complet.
    - Lipides : Huile d'olive, Avocat, Beurre de cacahuète, Amandes.
    - Légumes : Brocolis, Haricots verts, Épinards, Courgettes, Salade.
    2. EXCLUSION DE L'EXOTISME : Sauf demande explicite, bannis les ingrédients rares ou bizarres trouvés dans le dataset (pas de faisan, pas de chèvre, pas de pieds de poulet, pas d'abats).
    3. COHÉRENCE : Si le dataset contient des options complexes, ignore-les et utilise les options les plus "standard" pour un sportif.

    [CONSIGNE DE CALCUL ET FORMATAGE]
    Tu dois impérativement t'approcher de {total_target} kcal (+/- 100 kcal).
    Pour chaque repas, indique une estimation des calories pour prouver ton calcul.
    DEUX SUGGESTIONS : Pour chaque repas, propose une Option A (issue du dataset traduit) et une Option B (alternative classique).

    ### 1. BILAN NUTRITIONNEL CIBLE
    * **Total Calories** : {total_target} kcal / jour
    * **Répartition Macros suggérée** :
        - Protéines : {int(user['poids']) * 2}g (cible de 2g/kg pour cet objectif)
        - Lipides : {int(user['poids'])}g (environ 1g/kg)
        - Glucides : Le reste de la balance calorique.

    ### 2. MENU TYPE HEBDOMADAIRE
    Rempli les X par le nombre de calories correspondant aux repas.
    | Repas | Option A (Dataset Traduit) | kcal | Option B (Classique) | kcal |
    | :--- | :--- | :--- | :--- | :--- |
    | **Petit-Déjeuner** | [Nom A] | [X] kcal | [Nom B] | [X] kcal |
    | **Déjeuner** | [Nom A] | [X] kcal | [Nom B] | [X] kcal |
    | **Collation** | [Nom A] | [X] kcal | [Nom B] | [X] kcal |
    | **Dîner** | [Nom A] | [X] kcal | [Nom B] | [X] kcal |
    | **TOTAL ESTIMÉ** | **Cible : {total_target} kcal** | | **Cible : {total_target} kcal** | |

    ### 3. FICHES RECETTES DÉTAILLÉES
    Donne les quantités précises en grammes pour les deux options A et B de chaque repas.

    **[NOM DU REPAS] - OPTION A**
    * 🛒 **Ingrédients** : [Liste traduite avec poids en g]
    * 👨‍🍳 **Préparation** : [Étapes courtes en français]

    **[NOM DU REPAS] - OPTION B**
    * 🛒 **Ingrédients** : [Liste avec poids en g]
    * 👨‍🍳 **Préparation** : [Étapes courtes en français]

    ### 4. LISTE DES COURSES PAR RAYON
    * 🥦 **Légumes & Fruits** / 🥩 **Protéines** / 📦 **Épicerie**
    Tu dois Inclure les ingrédients nécessaires pour les deux options.

    [VÉRIFICATION FINALE]
    Si {constraints} n'est pas "Aucune", vérifie une dernière fois la sécurité. Assure-toi qu'AUCUN mot anglais ne subsiste.
    """

    # Appel du LLM
    response = llm.invoke(prisme_prompt)

    return {"nutrition_plan": response, "nutrition_revision_count": revision + 1}

# --- AGENT CRITIQUE SPORT (il va servir de vérification du contenu du programme de sport pour vérifier la cohérences des informations avec la demande de l'utilisateur) ---
def sport_critic_node(state: AgentState):
    print("\n🧐 [Auditeur Sport]: Analyse de la cohérence Sport/Objectif/Calories...")

    user = state["user_data"]
    plan = state["sport_plan"]
    e_cal = state["e_cal"]

    # Création du prompt qui va récupérer le plan générer par l'agent de sport pour vérifier la cohérence et l'exactitude des informations en validant ou non le programme
    critic_prompt = f"""
    Tu es un Expert en Sciences du Sport et Auditeur Qualité.

    [DONNÉES CLIENT]
    - Objectif : {user['objectif']}
    - Niveau : {user['niveau']}
    - Sport : {user['sport']}

    [PROGRAMME GÉNÉRÉ]
    - Dépense énergétique : {e_cal} kcal
    - Contenu :
    {plan}

    [TACHE]
    Vérifie la cohérence. Si c'est bon, réponds JUSTE "VALIDE".
    Si c'est mauvais, réponds "INVALID" suivi de l'explication.

    [RÉPONSE ATTENDUE]
    Commence ta réponse IMPÉRATIVEMENT par un seul mot : "VALIDE" ou "INVALID".
    """

    response = llm.invoke(critic_prompt)

    # --- PARSING ROBUSTE (FRANÇAIS/ANGLAIS) ---
    # On met tout en majuscule pour éviter les soucis de casse
    clean_resp = response.upper()

    # On cherche d'abord si c'est invalide (priorité à la sécurité)
    if "INVALID" in clean_resp or "INVALIDE" in clean_resp:
        is_valid = False

        # On nettoie pour ne garder que le message d'erreur
        feedback = response.replace("INVALID", "").replace("INVALIDE", "").replace("invalid", "").replace("invalide", "").strip()

        # Si le feedback commence par des caractères parasites (: , . -), on les retire
        feedback = feedback.lstrip(" :.-")

    # Sinon, on regarde si c'est valide (VALID ou VALIDE)
    elif "VALID" in clean_resp:
        is_valid = True
        feedback = "Validé par l'auditeur."

    # Cas de secours (si le LLM hallucine et ne met aucun mot clé on redemande à l'agent sport de générer un nouveau rapport même s'il était bon)
    else:
        is_valid = False
        feedback = f"Format de réponse critique incorrect : {response[:50]}..."

    if is_valid:
        print("✅ [Auditeur Sport]: Programme cohérent et validé.")
    else:
        print(f"❌ [Auditeur Sport]: Incohérence détectée -> {feedback}")

    return {"is_sport_valid": is_valid, "sport_feedback": feedback}

# ---AGENT CRITIQUE NUTRITION (il va lui vérifier la cohérence et la validité du programme de nutrition généré par l'agent de nutrition) ---
def nutrition_critic_node(state: AgentState):
    if not state.get("wants_nutrition"): return {"is_nutrition_valid": True}

    print("\n🧐 [Auditeur Nutrition]: Vérification des contraintes médicales...")

    plan = state.get("nutrition_plan")
    constraints = state.get("constraints_report", "Aucune")
    user_constraints = state["user_data"].get("details_nutrition", "Aucune")

    # Création du prompt en récupérant le plan nutrition pour le vérifier, si l'agent détecte un ingrédient présent dans la liste des contrainte, il rejette directement, sinon il vérifie la validité globale
    critic_prompt = f"""
    Tu es un Auditeur Sanitaire.

    [CONTRAINTES CLIENT]
    - Déclarées : {user_constraints}
    - Analyse risques : {constraints}

    [PLAN À VÉRIFIER]
    {plan}

    [TACHE]
    Rejette IMMÉDIATEMENT (INVALID) si un ingrédient interdit apparaît.
    Sinon réponds VALIDE.

    [RÉPONSE ATTENDUE]
    Commence ta réponse IMPÉRATIVEMENT par : "VALIDE" ou "INVALID".
    """

    response = llm.invoke(critic_prompt)

    # --- PARSING ROBUSTE ---
    clean_resp = response.upper()

    if "INVALID" in clean_resp or "INVALIDE" in clean_resp:
        is_valid = False
        feedback = response.replace("INVALID", "").replace("INVALIDE", "").replace("invalid", "").replace("invalide", "").strip()
        feedback = feedback.lstrip(" :.-")
    elif "VALID" in clean_resp:
        is_valid = True
        feedback = "Validé par l'auditeur."
    else:
        is_valid = False
        feedback = "Erreur de format critique nutrition."

    if is_valid:
        print("✅ [Auditeur Nutrition]: Plan sûr et validé.")
    else:
        print(f"❌ [Auditeur Nutrition]: Risque détecté -> {feedback}")

    return {"is_nutrition_valid": is_valid, "nutrition_feedback": feedback}


### Step 7: Conditional Routing & Dynamic Orchestration Logic
Defines graph edges, dynamic routing, and parallel execution logic.


In [ ]:
import copy
from typing import TypedDict, List, Literal
from langgraph.graph import StateGraph, END
from langchain_ollama import OllamaLLM

# Configure local Ollama LLM
llm = OllamaLLM(model="llama3")

# --- LOGIQUE DE ROUTAGE ---

# Dynamic routing between sport agents and parallel safety verification entre les agents sport et permet d'exécuter en parallèle l'agent des contraintes
def route_dynamic_orchestrator(state: AgentState):
    decision = state.get("router_decision", {})
    destinations = []

    # 1. Lancement de l'Expert Sport choisi
    sport_agent = decision.get("selected_sport_agent", "as_muscu")

    if sport_agent == "retry_input":
        return ["agent_ask_retry"]

    # Fallback safety checks sinon on redirige vers l'erreur sport
    if sport_agent == "error_sport":
        return ["error_sport"]

    if sport_agent not in ["as_muscu", "as_run", "as_cycle", "as_swim"]:
        sport_agent = "as_muscu"
    destinations.append(sport_agent)

    # 2. Lancement parallèle de l'Expert Santé (agent contrainte) si nécessaire
    if decision.get("activate_health_agent"):
        destinations.append("acont")
        state["has_constraints"] = True
    else:
        state["has_constraints"] = False

    # Note : L'info "activate_nutrition_loop" sera utilisée plus tard par le routeur de fin de sport

    return destinations

    # --- NOEUD D'ERREUR (permet de demander de re saisir les informations si le sport n'est pas pris en charge par le système) ---
def error_sport_node(state: AgentState):
    print("\n❌ [Système]: Désolé, ce sport n'est pas encore implémenté dans notre système.")
    print("👉 Merci de choisir parmi : Musculation, Running, Cyclisme ou Natation.")

    # Il y a un reset des questions seulement jusqu'au sport
    new_user_data = copy.deepcopy(state["user_data"]) # Utilisation de deecopy pour evité de modifier l'état des user_data déjà en place
    if "sport" in new_user_data: del new_user_data["sport"]

    # Décrementation des steps pour reposer les questions à l'utilisateur
    return {
        "user_data": new_user_data,
        "current_step": 4,
        "is_complete": False
    }

# --- LOGIQUE DE ROUTAGE APRÈS LA DÉCISION NUTRITIONNELLE ---

# Routage après la critique sportive
def route_after_sport_critic(state: AgentState):

    # 1. Est-ce qu'on avance ? (Validé ou Max essais)
    should_proceed = state["is_sport_valid"] or state["sport_revision_count"] >= 3

    if should_proceed:

        # On regarde la décision prise au début par le planificateur
        decision = state.get("router_decision", {})

        # On utilise aussi le fallback 'wants_nutrition' si le planificateur a échoué
        wants_nutri = decision.get("activate_nutrition_loop", state["wants_nutrition"])

        if wants_nutri:
            return "an_nutrition"
        # Si l'utilisateur ne veux pas de programme de nutrition le programme prend fin
        else:
            print("\n🏁 [Système]: Fin du coaching (Nutrition non requise).")
            return END

    # 2. Sinon, on boucle vers l'agent qui a été choisi
    decision = state.get("router_decision", {})
    return decision.get("selected_sport_agent", "as_muscu")


# B. Routage après critique NUTRITION
def route_after_nutrition_critic(state: AgentState):
    if state["is_nutrition_valid"]:
        return END # Fini !

    if state["nutrition_revision_count"] >= 3:
        print("⚠️ [Système]: Trop d'essais Nutrition. Fin.")
        return END

    return "an_nutrition"

# Routage après la génération du programme nutritionnel
def route_after_nutrition_gen(state):
    # Si l'agent nutrition n'a rien fait (car il attendait l'autre branche), on arrête là.
    # On détecte ça si 'nutrition_plan' est vide ou absent du changement récent.

    # On regarde si on est en "mode attente"
    # Si le sport n'est pas fini, c'est qu'on a fait un "return None" virtuel
    sport_done = state.get("is_sport_valid") or state.get("sport_revision_count", 0) >= 3
    constraints_ok = True
    if state.get("has_constraints"):
        constraints_ok = state.get("constraints_report") is not None

    # Si tout n'est pas prêt, on arrête le flux ICI (pour cette branche)
    if not sport_done or not constraints_ok:
        return END # On stoppe cette branche "fantôme"

    # Si tout est bon, on envoie au critique
    return "nutrition_critic"

#Routage pour compléter le questionnaire sans erreurs
def route_client_loop(state: AgentState):
    # Si le questionnaire est terminé, on passe au cerveau (Planificateur)
    if state["is_complete"]:
        return "agent_planner"
    # Sinon, on boucle sur le client pour la question suivante
    return "agent_client"


### Step 8: LangGraph Workflow Compilation
Assembles nodes and conditional edges into an executable state graph.


In [ ]:
# --- LANGGRAPH WORKFLOW COMPILATION ---

workflow = StateGraph(AgentState)

# Register agent nodes in graph pour pouvoir les utiliser
workflow.add_node("agent_client", agent_client_interactive)
workflow.add_node("agent_planner", agent_planner_node)
workflow.add_node("agent_ask_retry", agent_ask_retry_node)
workflow.add_node("error_sport", error_sport_node)
workflow.add_node("as_muscu", as_musculation_node)
workflow.add_node("as_run", as_running_node)
workflow.add_node("as_cycle", as_cyclisme_node)
workflow.add_node("as_swim", as_natation_node)
workflow.add_node("acont", acont_node)
workflow.add_node("sport_critic", sport_critic_node)
workflow.add_node("an_nutrition", an_nutrition_node)
workflow.add_node("nutrition_critic", nutrition_critic_node)

# Define workflow entry point (là où le programme va commencer)
workflow.set_entry_point("agent_client")

# 1. Client -> Planificateur
workflow.add_conditional_edges(
    "agent_client",
    route_client_loop,
    {
        "agent_client": "agent_client",   # On boucle pour poser la question suivante
        "agent_planner": "agent_planner"  # On avance quand tout est fini
    }
)

# 2. Planificateur -> Parallèlisme (Sport + Contraintes)
workflow.add_conditional_edges(
    "agent_planner",
    route_dynamic_orchestrator,
    ["as_muscu", "as_run", "as_cycle", "as_swim", "acont","error_sport","agent_ask_retry"]
)

# 5. Boucle de correction (Retry -> Retour au Planner)
# Une fois que l'utilisateur a corrigé, on renvoie au Planificateur pour vérification
workflow.add_edge("agent_ask_retry", "agent_planner")

workflow.add_edge("error_sport", "agent_client")

# Branche ACont (Ecrit et s'arrête, la donnée sera lue par Nutrition)
workflow.add_edge("acont", "an_nutrition")

# 2. Sport -> Critique Sport
for sport_node_name in ["as_muscu", "as_run", "as_cycle", "as_swim"]:
  workflow.add_edge(sport_node_name, "sport_critic")

# 3. Boucle Critique Sport
workflow.add_conditional_edges("sport_critic", route_after_sport_critic, {
    "as_muscu": "as_muscu",
    "as_run": "as_run",
    "as_cycle": "as_cycle",
    "as_swim": "as_swim",
    "an_nutrition": "an_nutrition",
    END: END
})

# 5. Nutrition -> Critique Nutrition
# On ne va au critique QUE si la nutrition a vraiment travaillé
workflow.add_conditional_edges(
    "an_nutrition",
    route_after_nutrition_gen,
    {
        "nutrition_critic": "nutrition_critic",
        END: END
    }
)

# 6. Boucle Critique Nutrition
workflow.add_conditional_edges("nutrition_critic", route_after_nutrition_critic, {
    "an_nutrition": "an_nutrition", # Retour correction
    END: END # Validation OK
})

app = workflow.compile()



### Step 9: Interactive Execution & End-to-End Evaluation
Run an interactive session or trigger a predefined user profile.


## Example User Profile
- **Name:** Marc
- **Gender:** Male
- **Weight:** 85 kg
- **Objective:** Sub-3h Marathon (Endurance conditioning)
- **Level:** Advanced
- **Frequency:** 4 sessions/week
- **Preferred Sport:** Running
- **Physical Condition:** Mild asthma
- **Nutrition Plan:** Yes
- **Dietary Constraints:** Gluten and lactose intolerance


In [ ]:
# --- WORKFLOW EXECUTION ---
import time
import re

print("--- SYSTÈME DE COACHING SMA ---")
start_time = time.time()
final_state = app.invoke({"user_data": {}, "current_step": 0, "is_complete": False})
end_time = time.time()
execution_duration = end_time - start_time

# System evaluation metrics
def print_metrics_report(final_state, duration):
    print("\n" + "="*60)
    print("📊 EVALUATION REPORT & SYSTEM METRICS")
    print("="*60)

    # --- 1. PERFORMANCE TEMPORELLE ---
    print(f"\n⏱️  LATENCY & EXECUTION TIME")
    print(f"   - Execution Time : {duration:.2f} secondes")

    # --- 2. CALORIC CONSISTENCY & BALANCE ---
    user = final_state.get("user_data", {})

    # A. On reconstitue le besoin théorique (Mathématiques)
    try:
        poids = float(user.get("poids", 70)) # Valeur par défaut si bug
    except:
        poids = 70.0

    genre = user.get("genre", "Homme").lower()

    # Estimation Métabolisme de Base (BMR) : ~24kcal/kg pour homme, ~22 pour femme
    facteur_base = 24 if "hom" in genre or "m" in genre else 22
    bmr = poids * facteur_base

    # Dépense Sportive (Calculée par les agents Sport)
    raw_cal = final_state.get("e_cal", 0)
    sport_burn = sum(raw_cal) if isinstance(raw_cal, list) else raw_cal

    # Ajustement selon l'objectif (Prise de masse = Surplus / Perte = Déficit)
    obj = user.get("objectif", "").lower()
    ajustement = 0
    if any(x in obj for x in ["masse", "muscle", "gros", "poids"]):
        ajustement = 300 # Surplus calorique léger
    elif any(x in obj for x in ["perte", "seche", "sèche", "maigrir", "fit"]):
        ajustement = -300 # Déficit calorique

    # LA CIBLE GLOBALE THÉORIQUE
    target_global = bmr + sport_burn + ajustement

    # B. On lit ce que l'IA a proposé dans le texte (Regex)
    plan_nutri = final_state.get("nutrition_plan", "")

    # On cherche un nombre proche de "kcal" ou "calories"
    match = re.findall(r"(\d+(?:\s\d+)*)\s*k?cal", plan_nutri, re.IGNORECASE)

    # Si le format "Total :" n'est pas trouvé, on cherche juste le plus gros nombre (souvent le total)
    print(f"\n⚖️  COHÉRENCE CALORIQUE (Global Journalier)")
    print(f"   - Métabolisme Base (Est.) : {bmr:.0f} kcal")
    print(f"   - Dépense Sport (CSV)     : +{sport_burn} kcal")
    print(f"   - Ajustement Objectif     : {ajustement:+d} kcal")
    print(f"   ------------------------------------------")
    print(f"   🎯 CIBLE GLOBALE CALCULÉE : {target_global:.0f} kcal")

    if match:
        clean_values=[]
        for val in match :
            clean_val=int(val.replace(" ", "").replace(",",""))
            clean_values.append(clean_val)

        generated_cal = max(clean_values)

        # Affichage des données de comparaison
        print(f"   🤖 MENU PROPOSÉ PAR IA    : {generated_cal} kcal")

        delta = abs(target_global - generated_cal)
        percent_error = (delta / target_global) * 100

        print(f"   - Écart Absolu            : {delta:.0f} kcal")
        print(f"   - Précision               : {100 - percent_error:.2f}% (Erreur: {percent_error:.2f}%)")

        if percent_error < 15:
            print("   ✅ VERDICT : COHÉRENT (L'IA respecte la physiologie)")
        else:
            print("   ⚠️ VERDICT : ÉCART NOTABLE (L'IA hallucine un peu)")
    else:
        print("   🤖 MENU PROPOSÉ PAR IA    : (Non détecté automatiquement)")
        print("   ❓ VERDICT : Impossible à mesurer (Format de texte inconnu)")

    # --- 3. EFFORT & ROBUSTESSE DU SYSTEME ---
    rev_sport = final_state.get("sport_revision_count", 0)
    rev_nutri = final_state.get("nutrition_revision_count", 0)

    print(f"\n🔄  AUTO-CORRECTIONS")
    print(f"   - Sport     : {rev_sport} itérations")
    print(f"   - Nutrition : {rev_nutri} itérations")

    # --- 4. ÉCONOMIE (COUT EN TOKENS EN SORTIE) ---
    # Récupération texte sport (liste ou str)
    plan_s = final_state.get("sport_plan", "")
    txt_s = " ".join(plan_s) if isinstance(plan_s, list) else str(plan_s)

    # On divise par 4 car une estimation est qu'un mot vaut environ 4 tokens
    tok_s = len(txt_s) // 4
    tok_n = len(plan_nutri) // 4
    total_tok = tok_s + tok_n

    # On prend un coût lambda de 2$/1M tokens même si ici on utilise une version gratuite (possible de le changer pour d'autre modèle)
    cost = (total_tok / 1000) * 0.002

    print(f"\n💰  ESTIMATION COÛT")
    print(f"   - Tokens Sport : ~{tok_s}")
    print(f"   - Tokens Nutrition : ~{tok_n}")
    print(f"   - Total Tokens en sortie: ~{total_tok}")
    print(f"   - Coût (Simulé): ${cost:.5f}")

    print("="*60 + "\n")

print_metrics_report(final_state, execution_duration)

print("\n" + "="*50 + "\nPROGRAMME FINAL\n" + "="*50)
print(f"\n[SPORT]\n{final_state['sport_plan']}")
if final_state.get('wants_nutrition'):
    print(f"\n[NUTRITION]\n{final_state['nutrition_plan']}")
else:
    print("\n[NUTRITION] Aucun programme demandé.")


--- SYSTÈME DE COACHING SMA ---
🤖 [Agent Client]: Comment t'appelles-tu ?
👤: MARC
🤖 [Agent Client]: Quel est ton genre (Homme/Femme) ?
👤: Homme
🤖 [Agent Client]: Quel est ton poids actuel (en kg) ?
👤: 85
🤖 [Agent Client]: Quel est ton objectif (Sèche, Prise de masse, etc.) ?
👤: courir un marathon en moins de 3h
🤖 [Agent Client]: Sport préféré ? (Laisse vide pour que je choisisse pour toi)
👤: 
🤖 [Agent Client]: Quel est ton niveau (Débutant, Intermédiaire, Avancé) ?
👤: Avancé
🤖 [Agent Client]: Combien de séances par semaine ?
👤: 4
🤖 [Agent Client]: As-tu des maladies ou conditions physiques (ex: problèmes cardiaques, blessures) ? (Sinon tape Non)
👤: asthme
🤖 [Agent Client]: Veux-tu un programme nutritionnel associé ? (Oui/Non)
👤: oui
🤖 [Agent Client]: As-tu des allergies/maladies ? (Sinon tape 'Non')
👤: oui, allergie au gluten et au lactose 

--- ✅ DONNÉES COLLECTÉES ---
- Nom: MARC
- Genre: Homme
- Poids: 85
- Objectif: courir un marathon en moins de 3h
- Sport: 
- Niveau: Avancé
- Fre

### Step 10: Multi-Agent vs. Monolithic LLM Comparative Benchmark
Evaluates the multi-agent graph against a single monolithic LLM prompt to quantify hallucination rates and constraint satisfaction.


In [ ]:
# --- MONOLITHIC AGENT (STRICT COMPARATIVE BASELINE) ---

def run_mono_agent_coaching_fair_comparison():
    print("--- 🤖 SYSTÈME DE COACHING (VERSION MONO-AGENT EXPERT) ---")
    print("      (Mêmes inputs + Mêmes consignes détaillées que le SMA)")

    # 1. COLLECTE (Identique SMA)
    questions = [
        ("nom", "Comment t'appelles-tu ?"),
        ("genre", "Quel est ton genre (Homme/Femme) ?"),
        ("poids", "Quel est ton poids actuel (en kg) ?"),
        ("objectif", "Quel est ton objectif (Sèche, Prise de masse, etc.) ?"),
        ("sport", "Sport préféré ?"),
        ("niveau", "Quel est ton niveau (Débutant, Intermédiaire, Avancé) ?"),
        ("frequence", "Combien de séances par semaine ?"),
        ("conditions_physiques", "As-tu des maladies ou conditions physiques ? (Sinon tape Non)"),
        ("wants_nutrition", "Veux-tu un programme nutritionnel associé ? (Oui/Non)"),
    ]

    user_data = {}
    for key, question in questions:
        response = input(f"🤖 {question}\n👤: ")
        user_data[key] = response.strip()

    wants_nutri = user_data["wants_nutrition"].lower() in ["oui", "o", "yes"]
    if wants_nutri:
        user_data["contraintes"] = input("🤖 As-tu des allergies/maladies ? (Sinon tape 'Non')\n👤: ")
    else:
        user_data["contraintes"] = "Non"

    if not user_data["sport"]:
        user_data["sport"] = "Running"
        print(f"⚠️ Sport par défaut : Running")

    # 2. OUTILS (Identique SMA)
    print("\n🛠️ [Système]: Appel des outils de calcul...")
    try:
        e_cal_real, _ = get_real_calories(user_data['sport'], float(user_data['poids']), user_data['niveau'])
        food_context = get_food_context(user_data['objectif']) if wants_nutri else "N/A"
    except:
        e_cal_real = 500
        food_context = "N/A"

    # 3. SÉLECTION DYNAMIQUE DES INSTRUCTIONS SPÉCIFIQUES
    # On récupère les consignes EXACTES des agents experts pour les injecter ici.

    sport_key = user_data['sport'].lower()

    # Consignes spécifiques selon le sport similaire au SMA
    if "muscu" in sport_key or "poids" in sport_key:
        specific_sport_instruction = """
        [EXPERTISE MUSCULATION]
        Tu dois agir comme un expert en biomécanique.
        - Chaque séance doit inclure : Nom, Temps, Exercices, Séries, Répétitions, Repos.
        - OBLIGATOIRE : Propose une variante "Sans Machine" pour chaque exercice.
        - Indique des conseils de charge (RPE ou %RM).
        - Intègre la surcharge progressive.
        """
    elif "run" in sport_key or "cour" in sport_key:
        specific_sport_instruction = """
        [EXPERTISE RUNNING]
        Tu dois agir comme un coach de niveau international (Périodisation).
        - Structure une progression sur plusieurs semaines (Base -> Puissance).
        - Détaille pour chaque sortie : Temps, Distance estimée, Allure (min/km).
        - OBLIGATOIRE : Inclus une séance de Renforcement Musculaire (PPG) par semaine.
        - Propose une alternative "Tapis de course" pour chaque séance.
        """
    elif "cycl" in sport_key or "vél" in sport_key:
        specific_sport_instruction = """
        [EXPERTISE CYCLISME]
        Tu es Directeur Sportif.
        - Équilibre le travail de puissance (FTP), vélocité et endurance.
        - Précise la Cadence (RPM) et l'intensité (Zones cardiaques).
        - OBLIGATOIRE : Propose une alternative Home-Trainer pour chaque séance.
        - Inclus des drills techniques (pédalage unijambiste, etc.).
        """
    elif "na" in sport_key or "swim" in sport_key:
        specific_sport_instruction = """
        [EXPERTISE NATATION]
        Tu es expert en hydrodynamique.
        - Structure la séance : Échauffement, Éducatifs (Drills), Corps de séance, Récupération.
        - Précise les distances en mètres et les temps de repos.
        - OBLIGATOIRE : Propose une alternative "Dry-land" (hors de l'eau).
        """
    else:
        specific_sport_instruction = "[EXPERTISE GÉNÉRALE] Crée un programme équilibré et progressif."

    # Consignes spécifiques Nutrition similaire au SMA
    if wants_nutri:
        specific_nutri_instruction = f"""
        [EXPERTISE NUTRITION - MÉTHODE PRISME]
        1. Calcule le besoin total (Base + {e_cal_real} kcal + Ajustement Objectif).
        2. Génère un Menu Hebdomadaire avec 2 options par repas.
        3. Fiches Recettes : Ingrédients précis, Instructions, Prix/5, Difficulté/5.
        4. Liste de Courses : Classée par rayons.
        5. SÉCURITÉ : Exclus strictement tout ingrédient lié à : "{user_data['contraintes']}".
        Utilise ces données réelles si possible : {food_context}
        """
    else:
        specific_nutri_instruction = "Ne génère PAS de programme nutritionnel."

    # 4. ASSEMBLAGE
    prompt_mono = f"""
    Tu es le "COACH ULTIME", une IA capable de remplacer une équipe entière d'experts.

    [PROFIL CLIENT]
    - Nom: {user_data['nom']}
    - Objectif: {user_data['objectif']}
    - Sport: {user_data['sport']} (Niveau {user_data['niveau']})
    - Fréquence: {user_data['frequence']} fois/semaine
    - Blessures: {user_data['conditions_physiques']}
    - Contraintes Alim: {user_data['contraintes']}

    [DONNÉES TECHNIQUES]
    - Dépense Sport : {e_cal_real} kcal (A mentionner obligatoirement).

    [TA MISSION]
    Génère une réponse unique contenant TOUT le coaching ci-dessous.

    --- PARTIE 1 : LE SPORT ---
    Applique scrupuleusement ces règles d'expert :
    {specific_sport_instruction}

    Adpate le volume au niveau {user_data['niveau']} et protège les blessures citées.

    --- PARTIE 2 : LA NUTRITION ---
    {specific_nutri_instruction}

    [FORMAT DE SORTIE]
    Utilise du Markdown clair. Sépare bien Sport et Nutrition.
    Sois exhaustif mais ne dépasse pas la longueur maximale autorisée.
    """

    print("\n" + "="*60)
    print("🔄 GÉNÉRATION MONO-AGENT (MEGA-PROMPT)...")
    print("="*60)

    start = time.time()
    response = llm.invoke(prompt_mono)
    end = time.time()

    print(response)

    # Métriques d'évaluation pour comparer avec le SMA
    tokens = len(response) // 4
    print("\n" + "-"*40)
    print("📊 MÉTRIQUES MONO-AGENT")
    print(f"⏱️ Temps : {end - start:.2f}s")
    print(f"💰 Coût  : ~{tokens} tokens")
    print("-"*40)

if __name__ == "__main__":
    run_mono_agent_coaching_fair_comparison()


--- 🤖 SYSTÈME DE COACHING (VERSION MONO-AGENT EXPERT) ---
      (Mêmes inputs + Mêmes consignes détaillées que le SMA)
🤖 Comment t'appelles-tu ?
👤: Marc
🤖 Quel est ton genre (Homme/Femme) ?
👤: Homme
🤖 Quel est ton poids actuel (en kg) ?
👤: 85
🤖 Quel est ton objectif (Sèche, Prise de masse, etc.) ?
👤: courir un marathon en moins de 3h
🤖 Sport préféré ?
👤: Course
🤖 Quel est ton niveau (Débutant, Intermédiaire, Avancé) ?
👤: Avancé
🤖 Combien de séances par semaine ?
👤: 4
🤖 As-tu des maladies ou conditions physiques ? (Sinon tape Non)
👤: asthme
🤖 Veux-tu un programme nutritionnel associé ? (Oui/Non)
👤: oui
🤖 As-tu des allergies/maladies ? (Sinon tape 'Non')
👤: oui, allergie au gluten et au lactose 

🛠️ [Système]: Appel des outils de calcul...

🔄 GÉNÉRATION MONO-AGENT (MEGA-PROMPT)...
**COACH ULTIME**

**PARTIE 1 : LE SPORT**

Bonjour Marc, pour atteindre ton objectif de courir un marathon en moins de 3h, je vais te proposer une progression sur plusieurs semaines (Base -> Puissance).

**Sema

### Automated End-to-End (E2E) Test Suite
Executes 5 stress-test scenarios covering standard flows, parallel allergy verification, single-sport light workflows, and guardrail error-correction triggers.


In [ ]:
import time
from unittest.mock import patch

# --- FONCTION UTILITAIRE DE TEST (CORRIGÉE LOGIQUE BOOLÉENNE) ---
def run_test(test_name, user_scenario, mock_inputs=None):
    print(f"\n{'='*80}")
    print(f"🧪 TEST SUITE: {test_name}")
    print(f"{'='*80}")

    # 1. CALCUL PRÉALABLE DES BOOLÉENS
    # Comme on saute l'Agent Client qui fait ce travail habituellement,
    # on doit le faire manuellement ici pour que le Routeur comprenne.
    wants_nutri_bool = str(user_scenario.get("wants_nutrition", "Non")).lower() in ["oui", "o", "yes", "true"]

    # Gestion souple des contraintes (si "Asthme", "Allergie" -> True / si "Non", "Ras" -> False)
    constraints_val = str(user_scenario.get("contraintes", "Non")).lower()
    has_constraints_bool = constraints_val not in ["non", "aucun", "aucune", "ras", "rien", "false"]

    # 2. ÉTAT INITIAL INTELLIGENT
    initial_state = {
        "user_data": user_scenario,
        "current_step": 99,       # On saute les questions
        "is_complete": True,      # On dit que c'est fini
        "wants_nutrition": wants_nutri_bool, # <--- La valeur correcte est injectée ici
        "has_constraints": has_constraints_bool, # <--- Et ici pour l'agent Santé
        "sport_revision_count": 0,
        "nutrition_revision_count": 0,
        "e_cal": 0
    }

    start_t = time.time()
    input_simulations = mock_inputs if mock_inputs else []

    with patch('builtins.input', side_effect=input_simulations):
        try:
            # On lance le graphe
            result = app.invoke(initial_state)
            status = "✅ SUCCESS"
        except Exception as e:
            result = None
            status = f"❌ ERREUR ({e})"
            # import traceback
            # traceback.print_exc()

    duration = time.time() - start_t

    # --- ANALYSE DES RÉSULTATS ---
    if result:
        print(f"\n📊 RÉSULTATS DU TEST :")
        decision = result.get("router_decision", {})
        sport_agent = decision.get("selected_sport_agent", "Inconnu")
        print(f"   - Agent Sportif Activé : {sport_agent}")
        print(f"   - Calories Calculées   : {result.get('e_cal')} kcal")

        # Vérif Nutrition
        nutri_plan = result.get("nutrition_plan", "")
        if wants_nutri_bool and nutri_plan:
             print(f"   - Nutrition            : GÉNÉRÉE ✅ ({len(str(nutri_plan))//4} tokens)")
        elif wants_nutri_bool and not nutri_plan:
             print(f"   - Nutrition            : FAILURE ⚠️ (Demandée mais vide)")
        else:
             print(f"   - Nutrition            : Ignorée (Logique respectée)")

        # Vérif Santé
        if has_constraints_bool:
            report = result.get("constraints_report", "")
            print(f"   - Agent Santé (ACont)  : {'Activé ✅' if report else 'FAILURE ❌'}")

        # Vérif Retry
        if "retry" in test_name.lower() and "user_data" in result:
             print(f"   - Sport Final Corrigé  : {result['user_data'].get('sport')}")

    print(f"\n🏁 Verdict : {status} (en {duration:.2f}s)")
    return result


### Test Scenarios Execution:


#### Scenario 1: Standard Flow (Inference + Running + Nutrition)


In [ ]:
# ------------------------------------------------------------------
# SCÉNARIO 1 : FLUX STANDARD (Déduction + Running + Nutrition + Conditions physique)
# ------------------------------------------------------------------
# Teste : Planificateur (Déduction), Agent Running, Agent Nutrition
user_1 = {
    "nom": "Alice", "genre": "Femme", "poids": 60, "niveau": "Intermédiaire", "frequence": 3, "conditions_physiques": "Asthme",
    "objectif": "Préparer un 10km",
    "sport": "", # VIDE -> Le planificateur doit déduire "Running"
    "wants_nutrition": "Oui", "contraintes": "Non"
}
res_1 = run_test("1. DÉDUCTION AUTOMATIQUE & RUNNING", user_1)



🧪 TEST SUITE: 1. DÉDUCTION AUTOMATIQUE & RUNNING

🧠 [Planificateur]: Analyse de la demande pour affectation de l'expert...
   📋 Ordre de mission en détail : {'selected_sport_agent': 'as_run', 'activate_health_agent': False, 'activate_nutrition_loop': True}

⏱️ [START] Agent Running (à 21:00:18

🏃 [Expert Running]: Génération du programme...
⏱️ [END] Agent Running fini à 21:00:34 (Durée: 15.47s)

🧐 [Auditeur Sport]: Analyse de la cohérence Sport/Objectif/Calories...
✅ [Auditeur Sport]: Programme cohérent et validé.

🥗 [Agent Nutrition]: Génération du plan...

🧐 [Auditeur Nutrition]: Vérification des contraintes médicales...
✅ [Auditeur Nutrition]: Plan sûr et validé.

📊 RÉSULTATS DU TEST :
   - Agent Sportif Activé : as_run
   - Calories Calculées   : 630 kcal
   - Nutrition            : GÉNÉRÉE ✅ (448 tokens)

🏁 Verdict : ✅ SUCCÈS (en 47.13s)


#### Scenario 2: Complex Parallel Flow (Strength Training + Celiac/Lactose Allergies + Parallel Safety)


In [ ]:
# ------------------------------------------------------------------
# SCÉNARIO 2 : FLUX COMPLEXE (Muscu + Allergie + Parallélisme)
# ------------------------------------------------------------------
# Teste : Agent Muscu, Agent Santé (ACont), Synchronisation
user_2 = {
    "nom": "Bob", "genre": "Homme", "poids": 80, "niveau": "Avancé", "frequence": 5, "conditions_physiques": "Non",
    "objectif": "Prise de masse",
    "sport": "Musculation",
    "wants_nutrition": "Oui",
    "contraintes": "Allergie aux arachides" # DOIT déclencher ACont
}
res_2 = run_test("2. CONTRAINTES DE SANTÉ & MUSCU", user_2)



🧪 TEST SUITE: 2. CONTRAINTES DE SANTÉ & MUSCU

🧠 [Planificateur]: Analyse de la demande pour affectation de l'expert...
   📋 Ordre de mission en détail : {'selected_sport_agent': 'as_muscu', 'activate_health_agent': True, 'activate_nutrition_loop': True}

⏱️ [START] Agent Contraintes à 21:01:08

⚠️ [Agent Contraintes]: Analyse en parallèle des restrictions : 'Allergie aux arachides'...

⏱️ [START] Agent Sport (à 21:01:08

🏋️ [Expert Musculation]: Génération du programme...
⏱️ [END] Agent Sport fini à 21:01:32 (Durée: 24.02s)
⏱️ [END] Agent Contraintes fini à 21:01:46 (Durée: 38.66s)

🧐 [Auditeur Sport]: Analyse de la cohérence Sport/Objectif/Calories...
✅ [Auditeur Sport]: Programme cohérent et validé.

🥗 [Agent Nutrition]: Génération du plan...

🧐 [Auditeur Nutrition]: Vérification des contraintes médicales...
✅ [Auditeur Nutrition]: Plan sûr et validé.

📊 RÉSULTATS DU TEST :
   - Agent Sportif Activé : as_muscu
   - Calories Calculées   : 520 kcal
   - Nutrition            : GÉNÉRÉE

#### Scenario 3: Lean Flow (Cycling + No Nutrition Plan)


In [ ]:
# ------------------------------------------------------------------
# SCÉNARIO 3 : FLUX LÉGER (Vélo + Pas de Nutrition)
# ------------------------------------------------------------------
# Teste : Agent Cycle, Court-circuitage de la nutrition
user_3 = {
    "nom": "Charlie", "genre": "Homme", "poids": 70, "niveau": "Débutant", "frequence": 2, "conditions_physiques": "Non",
    "objectif": "Balade santé",
    "sport": "Vélo",
    "wants_nutrition": "Non", # DOIT stopper après le sport
    "contraintes": "Non"
}
res_3 = run_test("3. CYCLISME SANS NUTRITION", user_3)




🧪 TEST SUITE: 3. CYCLISME SANS NUTRITION

🧠 [Planificateur]: Analyse de la demande pour affectation de l'expert...
   📋 Ordre de mission en détail : {'selected_sport_agent': 'as_cycle', 'activate_health_agent': False, 'activate_nutrition_loop': False}

⏱️ [START] Agent Cyclisme (à 21:03:40

🚴 [Expert Cyclisme]: Génération du programme...
⏱️ [END] Agent Cyclisme fini à 21:03:54 (Durée: 14.02s)

🧐 [Auditeur Sport]: Analyse de la cohérence Sport/Objectif/Calories...
✅ [Auditeur Sport]: Programme cohérent et validé.

🏁 [Système]: Fin du coaching (Nutrition non requise).

📊 RÉSULTATS DU TEST :
   - Agent Sportif Activé : as_cycle
   - Calories Calculées   : 336 kcal
   - Nutrition            : Ignorée (Logique respectée)

🏁 Verdict : ✅ SUCCÈS (en 19.70s)


#### Scenario 4: Aquatic Flow (Swimming Conditioning)


In [ ]:
# ------------------------------------------------------------------
# SCÉNARIO 4 : FLUX NATATION (Piscine)
# ------------------------------------------------------------------
# Teste : Agent Natation
user_4 = {
    "nom": "Dory", "genre": "Femme", "poids": 55, "niveau": "Intermédiaire", "frequence": 3, "conditions_physiques": "Non",
    "objectif": "Nager le crawl",
    "sport": "Natation",
    "wants_nutrition": "Oui", "contraintes": "Non"
}
res_4 = run_test("4. NATATION STANDARD", user_4)



🧪 TEST SUITE: 4. NATATION STANDARD

🧠 [Planificateur]: Analyse de la demande pour affectation de l'expert...
   📋 Ordre de mission en détail : {'selected_sport_agent': 'as_swim', 'activate_health_agent': False, 'activate_nutrition_loop': True}

⏱️ [START] Agent Natation (à 21:04:04

🏊 [Expert Natation]: Génération du programme...
⏱️ [END] Agent Natation fini à 21:04:21 (Durée: 17.43s)

🧐 [Auditeur Sport]: Analyse de la cohérence Sport/Objectif/Calories...
✅ [Auditeur Sport]: Programme cohérent et validé.

🥗 [Agent Nutrition]: Génération du plan...

🧐 [Auditeur Nutrition]: Vérification des contraintes médicales...
✅ [Auditeur Nutrition]: Plan sûr et validé.

📊 RÉSULTATS DU TEST :
   - Agent Sportif Activé : as_swim
   - Calories Calculées   : 350 kcal
   - Nutrition            : GÉNÉRÉE ✅ (981 tokens)

🏁 Verdict : ✅ SUCCÈS (en 70.02s)


#### Scenario 5: Safety Guardrail Trigger (Deliberate Error Injection & Automatic Revision Loop)


In [ ]:
# ------------------------------------------------------------------
# SCÉNARIO 5 : LE TEST ULTIME DU GARDE-FOU (Correction d'Erreur)
# ------------------------------------------------------------------
# Teste : Détection d'erreur sport -> Agent Retry -> Correction -> Relance
# Simulation : L'utilisateur entre "Tennis" (Invalide) -> Le système demande correction -> On simule "Running"
user_5_trap = {
    "nom": "Eve", "genre": "Femme", "poids": 65, "niveau": "Débutant", "frequence": 2, "conditions_physiques": "Non",
    "objectif": "Forme",
    "sport": "Tennis", # SPORT INVALIDE
    "wants_nutrition": "Oui", "contraintes": "Non"
}

# On prépare les réponses automatiques pour l'agent de correction
# Ordre des inputs demandés par agent_ask_retry_node :
# 1. Nouvel Objectif ? -> "Forme"
# 2. Nouveau Sport ?   -> "Running" (La correction valide)
# 3. Niveau ?          -> "Débutant"
# 4. Fréquence ?       -> "2"
mock_responses = ["Forme", "Running", "Débutant", "2"]

res_5 = run_test("5. ERREUR & AUTO-CORRECTION (TENNIS -> RUNNING)", user_5_trap, mock_inputs=mock_responses)



🧪 TEST SUITE: 5. ERREUR & AUTO-CORRECTION (TENNIS -> RUNNING)

🧠 [Planificateur]: Analyse de la demande pour affectation de l'expert...
   ⚠️ Sport non reconnu : 'tennis'. Demande de correction.

❌ [Système]: Le choix 'Tennis' n'est pas supporté ou incohérent.
   ⚠️ Pour garantir un bon programme, nous allons redéfinir tes paramètres sportifs.
   (Les experts dispos sont : Musculation, Running, Vélo, Natation)
🏃 Quel sport choisis-tu ? (Laisse vide pour que je choisisse selon l'objectif)

🔄 [Système]: Données mises à jour. Retour à l'analyse...

🧠 [Planificateur]: Analyse de la demande pour affectation de l'expert...
   📋 Ordre de mission en détail : {'selected_sport_agent': 'as_run', 'activate_health_agent': False, 'activate_nutrition_loop': True}

⏱️ [START] Agent Running (à 21:05:14

🏃 [Expert Running]: Génération du programme...
⏱️ [END] Agent Running fini à 21:05:27 (Durée: 12.6s)

🧐 [Auditeur Sport]: Analyse de la cohérence Sport/Objectif/Calories...
✅ [Auditeur Sport]: Programm